# 11 · Which checks reject healthy recordings (Stage R0)

## Purpose
Before any false alarm is attributed to a model, establish **which checks** reject healthy development recordings, and
**under what conditions** (speed, run). Exploratory analysis on development data only (CLAUDE.md §7, Stage R0): no
training, no new models, no scoring of recordings, no threshold tuning or re-derivation, and no fix. Reserved tests stay
sealed; `artifacts/bundle/`, `artifacts/final/` and `metadata.csv` are untouched; `results/final/` appears once, as
historical context only, never as a target.

**The protocol trap.** Thresholds are calibrated on calibration normals, so counting "false alarms" on those same
recordings is circular. Every number below is reported separately for the normals **used** to set a threshold (labelled
**in-sample**, not a false-alarm rate) and those **not used**. The healthy sample is **25 recordings from one folder**
(CLAUDE.md §5 rule 13): one recording moves a rate by 4 percentage points, and Clopper–Pearson intervals over 25 are wide.

Three evidence sources, each with its own models and its own thresholds, never mixed:
- **(A) Deployed bundle**: only its 5 calibration normals are scored (they set τ_e and τ_b): in-sample.
- **(B) Q1 development folds**: standalone B1s and AE-6 per outer fold; `test` normals are out of sample.
- **(C) Stage 3C LOCO runs (seed 42, 27 runs)**: the full rule (`ml.arbitrator.decide`, B_with_caution) and each check
  on its own; `eval` and unseen-fold normals are out of sample.

## Inputs
`metadata.csv` (development pool via `ml.splits.development_pool`, folders via `ml.splits.folder_proxy`);
`artifacts/bundle/manifest.json` (deployed thresholds, calibration-normal speed range) and
`artifacts/bundle/calibration_scores.csv`; `results/q1q3/q1_fit_cal_assignment.csv`,
`results/scores/b1s_nomic_s42_dev_scores.parquet`, `b1s_all_s42_dev_scores.parquet`, `ae6_s42_dev_channel_errors.parquet`;
`results/q1q3/q3_fit_cal_eval_assignment.csv`, `results/scores/cnn6ae6_s42_loco_decisions.parquet`,
`results/scores/b1s_nomic_loco_q3fit_scores.parquet`, `results/loco/decision_thresholds.csv`; `results/final/q1.json`
(context only). Config and split hashes are printed below.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> loads the metadata and the base configuration, prints the config and split hashes, selects the development-pool normals and checks their counts against CLAUDE.md §3 (49 normals in total, 25 in the development pool, one folder). It stops with an error if a count differs. It also reads the deployed thresholds and the calibration normals' speed range from the bundle manifest (read only).</div>

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ml.config import load_config, config_sha256
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR
from ml.splits import development_pool, folder_proxy, split_hash
from ml.rejection import attribute, per_recording, joint_patterns, standalone, rate, speed_band, CAUSES, OUTCOMES

ROOT = SYSTEM_CODE_DIR
cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(ROOT / "metadata.csv")
print("config sha256:", config_sha256(cfg)[:16], "| preprocess sha256:", config_sha256(cfg.preprocess)[:16], "| split hash:", split_hash(meta)[:16])
pool = development_pool(meta, cfg.development.allowed_closed, cfg.development.reserved_columns)
normals = pool[pool.class10 == "normal"].copy()
normals["folder"] = folder_proxy(normals)
assert (meta.class10 == "normal").sum() == 49 and len(normals) == 25, "normal counts differ from CLAUDE.md §3: stop and ask"
speed = normals.set_index("rec_id").fr_measured_hz
folder = normals.set_index("rec_id").folder
manifest = json.loads((ROOT / "artifacts/bundle/manifest.json").read_text())
THR = {k: manifest["thresholds"][k] for k in ("tau_p", "tau_d", "tau_e", "tau_b")}
CAL_LO, CAL_HI = manifest["data"]["calibration_normal_speed_range_hz"]
print(f"development normals: {len(normals)} (closed train {int((normals.split_closed == 'train').sum())}, closed val "
      f"{int((normals.split_closed == 'val').sum())}); folders: {normals.folder.value_counts().to_dict()}")
print(f"speeds {speed.min():.2f}-{speed.max():.2f} Hz; bundle calibration-normal speed range {CAL_LO:.2f}-{CAL_HI:.2f} Hz")
print("deployed thresholds:", {k: round(v, 6) for k, v in THR.items()})

## (A) Deployed bundle — in-sample only
The bundle's saved scores cover its 144 calibration recordings, 5 of them normal. Those 5 **set** τ_e and τ_b (each is the
maximum over them) and are among the 144 that set τ_p and τ_d, so they pass by construction: **in-sample, not a
false-alarm rate**. The 20 bundle-training normals have no saved bundle scores; this stage scores nothing, so they are
**not available** here.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> scores nothing: it takes the 5 calibration normals from the bundle's saved calibration scores, applies the deployed thresholds with the attribution functions (the full rule and each check), and shows one row per recording. Read the outcome column as in-sample.</div>

In [ ]:
cal_scores = pd.read_csv(ROOT / "artifacts/bundle/calibration_scores.csv")
A = attribute(cal_scores[cal_scores.class10 == "normal"].assign(**THR))
A["source"], A["group"] = "bundle_calibration", "calibration (in-sample)"
A[["rec_id", "top_class", "msp", "maha", "ae", "b1s", "outcome", "pattern", "tie_snapped"]]

## (B) Q1 development folds — standalone detectors
Each outer fold fits B1s and AE-6 on 11 normals, sets each threshold on 5–6 calibration normals (`cal`, in-sample) and
scores the remaining normals (`test`), so each of the 25 normals is a `test` normal exactly once. These are each detector
**on its own** against its own threshold; they are not the full decision (that is part C).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> reads the saved development-fold scores of the two B1s variants (vibration-only, as in the bundle; and all features) and of AE-6, keeps the normals, and counts alarms (score above the fold's own threshold) separately for the calibration normals (in-sample) and the test normals (out of sample), with 95 % Clopper–Pearson intervals over distinct recordings.</div>

In [ ]:
def q1_rows(name, path):
    d = pd.read_parquet(ROOT / path)
    d = d[d.class10 == "normal"]
    out = []
    for role, r in d.groupby("role"):
        out.append({"detector": name, "role": role, "group": "calibration (in-sample)" if role == "cal" else "not used (out of sample)",
                    "instances": len(r), "recordings": r.rec_id.nunique(), "alarms": int(r.alarm.sum()),
                    "alarm_recordings": sorted(r[r.alarm].rec_id), **{"rate": rate(r[r.alarm].rec_id.nunique(), r.rec_id.nunique())}})
    return out

B = pd.DataFrame(q1_rows("B1s (vibration only)", "results/scores/b1s_nomic_s42_dev_scores.parquet")
                 + q1_rows("B1s (all features)", "results/scores/b1s_all_s42_dev_scores.parquet")
                 + q1_rows("AE-6", "results/scores/ae6_s42_dev_channel_errors.parquet"))
B["rate_ci95"] = B["rate"].map(lambda r: f"{r['k']}/{r['n']} [{r['ci95'][0]:.2f}, {r['ci95'][1]:.2f}]")
B[["detector", "role", "group", "instances", "recordings", "alarms", "rate_ci95", "alarm_recordings"]]

## (C) Stage 3C LOCO runs — the full rule and each check
Per run: CNN-6 top class, MSP and Mahalanobis (shared) and AE-6 from the saved 3C decisions, B1s with its τ_b from the
saved B1s scores, and the run's τ_p, τ_d, τ_e from the saved thresholds. Roles: `cal` (set the run's thresholds; in-sample),
`eval` (seen-folder, not used) and `foldk_known` (the unseen outer fold, not used). A score within 1e-12 of its threshold
is a tie (accepted), as in the run that saved it (CLAUDE.md §7 R0 item 7).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> joins the three saved files per run and recording (one row per normal per run), applies the full rule (ml.arbitrator.decide, B_with_caution) and the per-check attribution, and marks each row as calibration (in-sample) or not used. It then checks that the attribution reproduces the saved 3C check flags and the saved rule-A decisions exactly, and counts the tie-snapped rows.</div>

In [ ]:
from ml.arbitrator import decide
dec = pd.read_parquet(ROOT / "results/scores/cnn6ae6_s42_loco_decisions.parquet")
b1 = pd.read_parquet(ROOT / "results/scores/b1s_nomic_loco_q3fit_scores.parquet")
th = pd.read_csv(ROOT / "results/loco/decision_thresholds.csv")
keys = ["rec_id", "withheld", "outer_fold", "role"]
all_rows = (dec.merge(b1[keys + ["b1s", "tau_b"]], on=keys, how="left", validate="one_to_one")
            .merge(th[["withheld", "outer_fold", "tau_p", "tau_d", "tau_e"]], on=["withheld", "outer_fold"], validate="many_to_one")
            .rename(columns={"pred": "top_class"}))
check_all = attribute(all_rows)
agree = {"msp_ok": bool((check_all.msp_ok == ~check_all.fail_msp).all()), "maha_ok": bool((check_all.maha_ok == ~check_all.fail_maha).all())}
ra = [decide("A", r.top_class, r.msp, r.maha, r.ae, r.b1s, {"tau_p": r.tau_p, "tau_d": r.tau_d, "tau_e": r.tau_e, "tau_b": r.tau_b})
      for r in check_all.itertuples()]
agree["rule_A_vs_saved"] = float(np.mean([(x["fault_class"] if x["decision"] == "known_fault" else x["decision"]) == s
                                          for x, s in zip(ra, check_all.decision_classifier_first)]))
C = check_all[check_all.class10 == "normal"].copy()
C["source"] = "loco_3c_s42"
C["group"] = np.where(C.role == "cal", "calibration (in-sample)", "not used (out of sample)")
print("agreement with the saved 3C files (all 11,548 rows):", agree)
print("tie-snapped rows: all", int((check_all.tie_snapped != "").sum()), "| normals", int((C.tie_snapped != "").sum()))
print("normals:", len(C), "instances,", C.rec_id.nunique(), "recordings; roles", C.role.value_counts().to_dict())

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> counts the full rule's four outcomes (plain healthy, healthy with caution, abstention, known fault) for each group, per instance and per distinct recording (a recording counts if it had that outcome in at least one run), with 95 % Clopper–Pearson intervals over recordings. Caution and abstention are kept apart.</div>

In [ ]:
rows = []
for g, x in C.groupby("group"):
    pr = per_recording(x)
    for o in OUTCOMES:
        rows.append({"group": g, "outcome": o, "instances": f"{int((x.outcome == o).sum())}/{len(x)}",
                     "recordings (ever)": f"{int((pr[o] > 0).sum())}/{len(pr)}",
                     "ci95 (recordings)": "[{:.2f}, {:.2f}]".format(*rate(int((pr[o] > 0).sum()), len(pr))["ci95"])})
    rows.append({"group": g, "outcome": "not plain healthy (any)", "instances": f"{int((x.outcome != 'healthy').sum())}/{len(x)}",
                 "recordings (ever)": f"{int(pr.ever_rejected.sum())}/{len(pr)}",
                 "ci95 (recordings)": "[{:.2f}, {:.2f}]".format(*rate(int(pr.ever_rejected.sum()), len(pr))["ci95"])})
outcomes_table = pd.DataFrame(rows)
outcomes_table

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> shows, for the normals not used by a run, each check on its own against its own threshold (how many instances and recordings fail it) next to the full rule's result, so that 'a detector alarmed' is not read as 'the system did not return a clean healthy verdict'.</div>

In [ ]:
N = C[C.group == "not used (out of sample)"]
sa = standalone(N)
standalone_table = pd.DataFrame([{"check": k, "fails (instances)": f"{v['instances']['k']}/{v['instances']['n']}",
                                  "fails (recordings)": f"{v['recordings']['k']}/{v['recordings']['n']}"} for k, v in sa.items()])
full = {"full rule: not plain healthy (instances)": f"{int((N.outcome != 'healthy').sum())}/{len(N)}",
        "full rule: abstention": f"{int((N.outcome == 'abstention').sum())}/{len(N)}",
        "full rule: caution": f"{int((N.outcome == 'healthy_with_caution').sum())}/{len(N)}"}
print(full)
standalone_table

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> lists the joint patterns of failed checks behind every non-accepted verdict (which combinations occur, how often, in how many recordings), for the not-used normals and, separately, the calibration (in-sample) normals. It also counts, among abstentions, how often each check is part of the pattern.</div>

In [ ]:
jp_out = joint_patterns(N)
jp_cal = joint_patterns(C[C.group == "calibration (in-sample)"])
ab = N[N.outcome == "abstention"]
in_abstentions = {k: f"{int(ab[f'fail_{k}'].sum())}/{len(ab)}" for k in CAUSES}
ca = N[N.outcome == "healthy_with_caution"]
in_cautions = {k: f"{int(ca[f'fail_{k}'].sum())}/{len(ca)}" for k in ("ae", "b1s")}
print("abstentions containing each failed check:", in_abstentions)
print("cautions containing each failed check:", in_cautions)
display(jp_out)
jp_cal

## Breakdown by speed and by run
All 25 normals come from **one folder** (`normal/-`), so a folder breakdown is degenerate; the run (outer fold, withheld
class) is the grouping available instead. Speed is compared with the bundle's calibration-normal speed range (from the
manifest) and with each run's own calibration normals (2–3 per run).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> builds one row per normal (not-used instances only): its measured speed, whether it lies below, inside or above the bundle's calibration-normal speed range, and how often each outcome occurred across its runs. It then groups the not-used instances by whether the recording's speed lies inside the speed range of that run's own calibration normals.</div>

In [ ]:
per = per_recording(N).set_index("rec_id")
per["speed_hz"] = speed
per["vs_bundle_cal_range"] = speed_band(per.speed_hz, CAL_LO, CAL_HI)
per["not_plain_healthy_share"] = 1 - per.healthy / per.instances
run_cal = (C[C.role == "cal"].assign(s=lambda d: d.rec_id.map(speed)).groupby(["withheld", "outer_fold"]).s.agg(["min", "max"]))
Nr = N.join(run_cal, on=["withheld", "outer_fold"]).assign(s=lambda d: d.rec_id.map(speed))
Nr["vs_run_cal_range"] = np.where(Nr.s < Nr["min"], "below", np.where(Nr.s > Nr["max"], "above", "inside"))
by_band = per.groupby("vs_bundle_cal_range").agg(recordings=("instances", "size"), instances=("instances", "sum"),
                                                 not_plain_healthy=("healthy", lambda s: int((per.loc[s.index, "instances"] - s).sum())))
by_band["share"] = (by_band.not_plain_healthy / by_band.instances).round(3)
by_run_band = Nr.groupby("vs_run_cal_range").outcome.value_counts().unstack(fill_value=0)
by_run_band["share_not_plain_healthy"] = (1 - by_run_band["healthy"] / by_run_band.sum(axis=1)).round(3)
display(by_band)
display(by_run_band)
per.sort_values("speed_hz")[["speed_hz", "vs_bundle_cal_range", "instances", "healthy", "healthy_with_caution", "abstention",
                             "known_fault", "not_plain_healthy_share"] + [f"ever_fail_{k}" for k in CAUSES]].round(3)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> plots, for each normal (not-used instances), the share of its runs in which it was not accepted as plain healthy, split into caution and abstention, against its measured speed (Hz); the shaded band is the bundle's calibration-normal speed range. The figure is saved to results/figures/r0_rejection_by_speed.png.</div>

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.0))
p = per.sort_values("speed_hz")
ax.axvspan(CAL_LO, CAL_HI, color="#e3f2fd", label="bundle calibration-normal speed range")
ax.bar(p.speed_hz, p.abstention / p.instances, width=0.45, color="#5b5bd6", label="abstention (share of runs)")
ax.bar(p.speed_hz, p.healthy_with_caution / p.instances, width=0.45, bottom=p.abstention / p.instances, color="#e0a030",
       label="healthy with caution (share of runs)")
ax.set_xlabel("measured shaft speed (Hz)")
ax.set_ylabel("share of runs not plain healthy")
ax.set_ylim(0, 1.02)
ax.set_title("Healthy development normals (25, one folder), Stage 3C runs, instances not used by the run")
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)
fig.tight_layout()
(ROOT / "results/figures").mkdir(parents=True, exist_ok=True)
fig.savefig(ROOT / "results/figures/r0_rejection_by_speed.png", dpi=150)
plt.show()

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> groups the not-used instances by run: by outer fold and role, and by withheld class, to see whether rejections concentrate in particular runs.</div>

In [ ]:
by_fold = N.groupby(["outer_fold", "role"]).outcome.value_counts().unstack(fill_value=0)
by_withheld = N.groupby("withheld").outcome.value_counts().unstack(fill_value=0)
display(by_fold)
by_withheld

## Historical context (never a target)
The final evaluation (F1) is shown only to place these development findings next to what was already reported; nothing
here is tuned on it, and it is not used for any conclusion.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> reads the saved final-evaluation Q1 summary (results/final/q1.json) and prints, for each reserved test, how many healthy recordings the primary rule did not call healthy. Context only; nothing is fitted or tuned on it.</div>

In [ ]:
f1 = json.loads((ROOT / "results/final/q1.json").read_text())["bundle"]
f1_context = {t: {"normals": v["n_normals"], "false_alarm": v["rules"]["B_with_caution"]["false_alarm"]["k"]} for t, v in f1.items()}
f1_context

## Findings
Every rate below comes from **25 development normals from one folder** (CLAUDE.md §5 rule 13); one recording moves a
rate by 4 points, and the intervals are 95 % Clopper–Pearson over recordings.

- **In-sample numbers are clean by construction; out-of-sample numbers are not.** The bundle's 5 calibration normals are
  5/5 plain healthy and the Q1 calibration normals raise 0/17 alarms (they set the thresholds). In the Stage 3C runs the
  calibration normals are 64/71 instances plain healthy (0 cautions). The normals **not used** by a run are not plain
  healthy in **101/299 instances (34 %)**, from **19/25 recordings** at least once [0.55, 0.91].
- **Standalone detectors versus the full rule (not-used normals).** On its own each check fails: MSP 28/299 instances
  (7 recordings), Mahalanobis 47/299 (11), AE-6 73/299 (16), B1s 69/299 (14). The full rule turns this into
  **49 abstentions** (11 recordings) and **52 cautions** (16 recordings), never a known fault. In the Q1 development folds,
  where each normal is a test normal once, B1s (vibration only) alarms on **6/25** [0.09, 0.45], B1s (all features) on
  5/25 [0.07, 0.41] and AE-6 on **2/25** [0.01, 0.26].
- **Abstention and caution are different failures.** Abstentions contain a failed Mahalanobis check in **47/49**
  instances (MSP in 28/49), and in **30/49** Mahalanobis, AE-6 and B1s all fail together: the recording is atypical to
  every model at once. Cautions are split almost evenly: **B1s alone 18, AE-6 alone 17, both 17** of 52.
- **Speed matters more than any single model.** The 4 normals below the bundle's calibration-normal speed range
  (18.08–38.34 Hz; they run at 12.9–15.9 Hz) and the 1 above it (39.5 Hz) are not plain healthy in **39/50** and
  **10/12** of their runs (78 %, 83 %), against **52/237 (22 %)** for the 20 inside it. Against each run's own 2–3
  calibration normals: below 77 %, above 32 %, inside 19 %. Inside the range, a few recordings carry most rejections
  (normal__27.4432: 0/10 runs plain healthy; normal__32.9728 5/11; 18–20 Hz and 38.3 Hz normals about half). Speed and
  recording identity are confounded here (each speed is one recording), and only 5 recordings lie outside the range.
- **Attribution: the healthy rejections are not attributable to a single model.** Abstentions follow the Mahalanobis
  check, but it co-fails with both healthy-baseline detectors in most of them; cautions divide between AE-6 and B1s;
  and the strongest pattern is the operating speed relative to the few calibration normals, together with a handful of
  individual recordings. **It does not separate by model.**
- **Hypotheses for a later stage (not tested here).** Thresholds τ_e and τ_b are maxima over only 2–5 calibration
  normals covering a narrow speed range, so healthy recordings at other speeds exceed them; the Mahalanobis familiarity
  check may be speed-sensitive in the same way. The final evaluation (historical context only) is consistent with this
  (5/7 and 19/20 reserved normals not called healthy, mostly outside the training and calibration speed ranges), but it is
  not used for any conclusion.


<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> writes the outputs: one row per scored healthy instance of the full-rule sources (A and C) with its scores, thresholds, failed checks, outcome, joint pattern, group, speed and folder (r0_attribution.csv); one row per healthy recording summarising its not-used instances (r0_attribution_by_recording.csv); and the JSON summary.</div>

In [ ]:
out_dir = ROOT / "results/research"
out_dir.mkdir(parents=True, exist_ok=True)
cols = ["source", "group", "withheld", "outer_fold", "role", "rec_id", "top_class", "msp", "maha", "ae", "b1s", "tau_p", "tau_d",
        "tau_e", "tau_b", *[f"fail_{k}" for k in CAUSES], "pattern", "decision", "caution", "outcome", "rule_causes", "tie_snapped"]
att = pd.concat([A.assign(withheld=None, outer_fold=None, role="calibration"), C], ignore_index=True).reindex(columns=cols)
att["speed_hz"], att["folder"] = att.rec_id.map(speed), att.rec_id.map(folder)
att.to_csv(out_dir / "r0_attribution.csv", index=False)
per.reset_index().to_csv(out_dir / "r0_attribution_by_recording.csv", index=False)
summary = {
    "stage": "R0", "label": "exploratory; development data only; reserved tests sealed; no training, scoring or threshold change",
    "healthy_sample": {"recordings": 25, "folders": 1, "note": "25 development normals from one folder (normal/-); one recording = 4 percentage points"},
    "config_sha256": config_sha256(cfg), "split_hash": split_hash(meta),
    "A_bundle_in_sample": {"normals": len(A), "outcomes": A.outcome.value_counts().to_dict(), "thresholds": THR,
                           "not_available": "the 20 bundle-training normals have no saved bundle scores"},
    "B_q1_dev_standalone": B.drop(columns=["rate_ci95"]).to_dict(orient="records"),
    "C_loco_full_rule": {"instances": len(C), "recordings": int(C.rec_id.nunique()), "agreement_with_saved_3c": agree,
                         "tie_snapped_normal_instances": int((C.tie_snapped != "").sum()),
                         "outcomes": outcomes_table.to_dict(orient="records"),
                         "standalone_not_used": sa, "abstentions_containing_check": in_abstentions, "cautions_containing_check": in_cautions,
                         "joint_patterns_not_used": jp_out.to_dict(orient="records"), "joint_patterns_in_sample": jp_cal.to_dict(orient="records")},
    "speed": {"bundle_calibration_normal_range_hz": [CAL_LO, CAL_HI], "by_bundle_range": by_band.reset_index().to_dict(orient="records"),
              "by_run_calibration_range": by_run_band.reset_index().to_dict(orient="records")},
    "historical_context_final_q1": f1_context,
}
(out_dir / "r0_rejection_analysis.json").write_text(json.dumps(summary, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("written:", sorted(p.name for p in out_dir.iterdir()), "+ results/figures/r0_rejection_by_speed.png")

## Outputs written
- `results/research/r0_attribution.csv`: one row per scored healthy instance of sources A and C (scores, thresholds,
  failed checks, joint pattern, outcome, group, run, speed, folder).
- `results/research/r0_attribution_by_recording.csv`: one row per healthy recording (its not-used Stage 3C instances).
- `results/research/r0_rejection_analysis.json`: the summary above.
- `results/figures/r0_rejection_by_speed.png`: rejection share by measured speed.